# GEE からのデータ読み込みと可視化

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astrocamp-2026-siaPpts/astrocamp-2026-sia/blob/main/tutorials/00_example_gee.ipynb)

このノートブックでは Google Earth Engine (GEE) を使って以下の操作を行う。

1. GEE の認証と初期化
2. Sentinel-2 衛星画像の読み込み（日付・範囲でフィルタ）
3. NDVI（正規化植生指数）の計算
4. geemap を使った地図上での可視化
5. 画像のプロパティ確認とエクスポート

**実行前に済ませておくこと:** [GEE アカウント登録](https://earthengine.google.com/) が完了していること。

In [1]:
# 必要なライブラリのインストール（Colab / ローカルの Jupyter どちらでも同じ）
!pip install geemap earthengine-api


In [2]:
import ee
import geemap

# ---- GEE の認証・初期化（Colab / ローカル両対応） ----
# 既に認証済みならブラウザは開かず、既存の認証を使います。初回のみブラウザが開きます。
try:
    from env import gee_setup
    gee_setup()
except ImportError:
    # env.py 未アップロード時のフォールバック（従来のコード）
    ee.Authenticate()
    try:
        ee.Initialize()
    except Exception:
        ee.Initialize(project='ee-yourproject')  # 自分のプロジェクトID に置き換える


GEE 初期化成功（既存の認証を利用）


In [3]:
# 解析対象の範囲を設定する（福島県浜通り周辺）
roi = ee.Geometry.Rectangle([140.5, 37.2, 141.2, 37.8])

# Sentinel-2 コレクションを読み込む
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2024-04-01', '2024-09-30')  # 2024年の生育期
    .filterBounds(roi)                        # 対象範囲
    .filterMetadata('CLOUDY_PIXEL_PERCENTAGE', 'less_than', 20)  # 雲20%未満
)

print(f'該当画像数: {collection.size().getInfo()}')

# 最も雲の少ない画像を1枚選ぶ
image = collection.sort('CLOUDY_PIXEL_PERCENTAGE').first()
print(f'選択画像: {image.id().getInfo()}')
print(f'撮影日時: {image.date().format("YYYY-MM-dd").getInfo()}')

該当画像数: 23
選択画像: 20240503T011649_20240503T011652_T54SWG
撮影日時: 2024-05-03


In [4]:
# NDVI = (NIR - Red) / (NIR + Red)
# Sentinel-2 L2A: B8 = NIR, B4 = Red
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# NDVI の統計情報を確認する
stats = ndvi.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=100,
    bestEffort=True
)
print(f'NDVI 範囲: {stats.getInfo()}')

NDVI 範囲: {'NDVI_max': 0.8807157057654076, 'NDVI_min': -0.2602100350058343}


In [5]:
# geemap で地図を作成する
m = geemap.Map()

# 真色画像（RGB）を追加
vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 2000}
m.add_layer(image, vis_rgb, 'True Color (RGB)')

# NDVI を追加
vis_ndvi = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}
m.add_layer(ndvi, vis_ndvi, 'NDVI')

# 地図の中心を対象範囲に設定する
m.center_object(roi)
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [6]:
# 画像のメタデータを確認する
props = image.propertyNames()
print('=== 画像プロパティ一覧 ===')
for p in props.getInfo()[:10]:  # 最初の10件のみ表示
    val = image.get(p).getInfo()
    print(f'{p}: {val}')

print('\n=== 利用可能なバンド ===')
bands = image.bandNames()
for b in bands.getInfo():
    print(b)

=== 画像プロパティ一覧 ===
system:version: 1789039634821333
system:id: COPERNICUS/S2_SR_HARMONIZED/20240503T011649_20240503T011652_T54SWG
SPACECRAFT_NAME: Sentinel-2B
SATURATED_DEFECTIVE_PIXEL_PERCENTAGE: 0
BOA_ADD_OFFSET_B12: -1000
CLOUD_SHADOW_PERCENTAGE: 0.071649
system:footprint: {'type': 'LinearRing', 'coordinates': [[140.99965911947453, 37.94760083694939], [140.99965825600265, 37.947589572013825], [140.99966275712455, 36.95806138321374], [140.9997085121117, 36.9580199557646], [140.99974742377003, 36.957974191549226], [140.99976588493317, 36.95797139775334], [142.2327821481976, 36.95157209139592], [142.23283459575774, 36.951608113804866], [142.23289231243857, 36.95163862216367], [142.2328960413269, 36.951653338363045], [142.2409686635663, 37.446330687532146], [142.2492407958712, 37.94095745368693], [142.24919514676603, 37.9409994125808], [142.24915655180897, 37.941045551331165], [142.24913782191106, 37.94104854272425], [140.99977237463816, 37.9476795408181], [140.9997198299243, 37.94764300

In [7]:
# （ローカルで実行しても、出力先は自分の Google Drive になります）
# 画像を GeoTIFF として Google Drive にエクスポートする
# ※ 実行前に Google Drive の空き容量を確認すること

task = ee.batch.Export.image.toDrive(
    image=ndvi.clip(roi),
    description='NDVI_export',
    folder='GEE_exports',
    fileNamePrefix='ndvi_fukushima',
    region=roi,
    scale=100,
    crs='EPSG:4326',
    fileFormat='GeoTIFF'
)
task.start()
print('エクスポートタスクを開始しました。')
print('Google Drive → GEE_exports フォルダに保存されます。')

エクスポートタスクを開始しました。
Google Drive → GEE_exports フォルダに保存されます。
